# 7.1 Capstone: Build a Mini RAG Chatbot

Interactive notebook for the **How AI Works** course. Run each cell top to bottom (Shift+Enter). The text and explanations live in the lesson page.

In [ ]:
# Setup: install the one dependency this course uses.
%pip install numpy --quiet

In [ ]:
documents = [
    "The dot product measures how aligned two vectors are.",
    "Embeddings turn words and sentences into dense vectors of numbers.",
    "Attention lets a transformer weigh the importance of each token.",
    "Softmax turns a list of scores into probabilities that sum to one.",
    "RAG retrieves relevant documents and adds them to the prompt.",
    "An agent uses tools in a loop to take actions in the world.",
]

print(f"Knowledge base has {len(documents)} documents.")

In [ ]:
import numpy as np
import re

# Common words that carry little meaning; we ignore them when embedding.
STOPWORDS = {
    "the", "a", "an", "of", "to", "and", "is", "are", "in", "on", "how",
    "does", "do", "what", "it", "that", "this", "with", "for", "as", "into",
}

def tokenize(text):
    # Lowercase, split into words, and drop stopwords (see Module 2.1 Tokenization).
    return [w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOPWORDS]

# Build a vocabulary from every meaningful word in the knowledge base.
vocabulary = sorted({word for doc in documents for word in tokenize(doc)})
word_to_index = {word: i for i, word in enumerate(vocabulary)}

def embed(text):
    """Turn text into a word-count vector over the fixed vocabulary."""
    vector = np.zeros(len(vocabulary))
    for word in tokenize(text):
        if word in word_to_index:
            vector[word_to_index[word]] += 1.0
    return vector

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Words matched in 'dot product of vectors': {int(embed('dot product of vectors').sum())}")

In [ ]:
# Pre-compute an embedding for every document once.
document_embeddings = np.array([embed(doc) for doc in documents])

def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return 0.0 if denom == 0 else float(np.dot(a, b) / denom)

def retrieve(question, top_k=2):
    """Return the top_k most similar documents to the question."""
    query_vector = embed(question)
    scores = [cosine_similarity(query_vector, doc_vec) for doc_vec in document_embeddings]
    ranked = sorted(range(len(documents)), key=lambda i: scores[i], reverse=True)
    return [(documents[i], scores[i]) for i in ranked[:top_k]]

question = "How does an attention mechanism use scores?"
for doc, score in retrieve(question):
    print(f"{score:.3f}  {doc}")

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful tutor. Answer using ONLY the provided context. "
    "If the context doesn't contain the answer, say you don't know."
)

def build_prompt(question, retrieved):
    context = "\n".join(f"- {doc}" for doc, _ in retrieved)
    return (
        f"[SYSTEM]\n{SYSTEM_PROMPT}\n\n"
        f"[CONTEXT]\n{context}\n\n"
        f"[QUESTION]\n{question}"
    )

print(build_prompt(question, retrieve(question)))

In [ ]:
def generate_offline(prompt, retrieved):
    """A stand-in for an LLM: return the top retrieved fact as the 'answer'."""
    if not retrieved or retrieved[0][1] == 0.0:
        return "I don't know based on the provided context."
    return f"Based on the context: {retrieved[0][0]}"

retrieved = retrieve(question)
print(generate_offline(build_prompt(question, retrieved), retrieved))

In [ ]:
def chatbot(question, generate=generate_offline):
    retrieved = retrieve(question)
    prompt = build_prompt(question, retrieved)
    return generate(prompt, retrieved)

for q in [
    "What does softmax do?",
    "How do agents take actions?",
    "What is the capital of France?",
]:
    print(f"Q: {q}")
    print(f"A: {chatbot(q)}\n")

In [ ]:
def generate_with_claude(prompt, retrieved, model="claude-opus-4-8"):
    """Drop-in replacement for generate_offline that calls a real Claude model.

    Requires `pip install anthropic` and the ANTHROPIC_API_KEY environment
    variable. The import is lazy so this lesson still runs without the package.
    """
    import anthropic  # imported here so the rest of the lesson runs offline

    context = "\n".join(f"- {doc}" for doc, _ in retrieved)
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    response = client.messages.create(
        model=model,                # the latest, most capable Claude model
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {prompt}"}],
    )
    return "".join(block.text for block in response.content if block.type == "text")

# To use it, just pass it in:  chatbot("What does softmax do?", generate=generate_with_claude)
print("generate_with_claude() is ready — set ANTHROPIC_API_KEY, then pass it to chatbot().")

## Exercises

<details>
<summary>Show solution</summary>

**1. Add a document and re-query.** Add a new fact about "neurons" to the
`documents` list, rebuild the vocabulary and embeddings, and ask a question about
neurons.

```python
documents.append("A neuron computes a weighted sum of its inputs and applies an activation.")
vocabulary = sorted({word for doc in documents for word in tokenize(doc)})
word_to_index = {word: i for i, word in enumerate(vocabulary)}
document_embeddings = np.array([embed(doc) for doc in documents])
print(retrieve("what does a neuron compute?", top_k=1))
```

Expected: the new neuron document is retrieved as the best match.
</details>

<details>
<summary>Show solution</summary>

**2. Change `top_k`.** Make `retrieve` return the top 3 documents instead of 2 and
re-run the prompt builder. How does the assembled context change?

```python
retrieved = retrieve("How does an attention mechanism use scores?", top_k=3)
print(len(retrieved), "documents retrieved")
```

Expected: `3 documents retrieved`, and `build_prompt` now lists three context
lines. More context can help the model but also fills the
[context window](../glossary.md#context-window) faster.
</details>

<details>
<summary>Show solution</summary>

**3. Raise the bar for "I don't know".** Modify `generate_offline` so it answers
only when the top similarity score is above `0.2`; otherwise it says it doesn't
know. Test it with an unrelated question.

```python
def generate_offline(prompt, retrieved, threshold=0.2):
    if not retrieved or retrieved[0][1] < threshold:
        return "I don't know based on the provided context."
    return f"Based on the context: {retrieved[0][0]}"

print(chatbot("Tell me about the weather on Mars."))
```

Expected: `I don't know based on the provided context.` — a higher threshold makes
the bot more cautious about weak matches.
</details>

---

**That's the whole stack — congratulations!** You started with three numbers in a
vector and finished with a working retrieval-augmented chatbot. From here, try
swapping in a real embedding model, growing the knowledge base, or giving your
agent a second tool. You now understand, end to end, how modern AI works.

*Try each exercise in a new cell below before opening the solution.*